# Sesión 20 — PySpark DataFrames I: lectura, transformaciones y acciones

**Asignatura:** Big Data — Ciencia de Datos, Universidad Externado de Colombia  
**Tipo de actividad:** Presencial · Guiada en clase (formativa)  
**Objetivo:** Familiarizarse con la sintaxis y las operaciones fundamentales de PySpark mediante la exploración de un dataset real.

---

**Prerrequisitos:**
- Conceptos de Apache Spark: driver, executors, DAG, lazy evaluation.
- Experiencia previa en Python y pandas.

**Al finalizar esta sesión, el estudiante será capaz de:**
1. Crear una SparkSession sobre el clúster Hadoop/YARN.
2. Cargar un archivo CSV como DataFrame de Spark desde HDFS.
3. Aplicar transformaciones y acciones básicas sobre un DataFrame.

> **Nota:** Esta es la primera de dos sesiones dedicadas a PySpark DataFrames. En la Sesión 21 se cubrirán agrupaciones, agregaciones, ordenamientos y escritura de resultados.


## 1. Configuración de la SparkSession en el clúster

En este entorno, PySpark ya viene preinstalado como parte de la instalación de Hadoop/Spark en el nodo maestro; no es necesario instalarlo ni configurar un entorno Java embebido como se haría en Colab. El siguiente paso es crear la SparkSession que se conectará al clúster real a través de YARN.


### ¿Qué es la SparkSession?

En las sesiones previas se estudió la arquitectura de Spark: un **driver** que coordina el trabajo y **executors** que lo ejecutan en paralelo. La `SparkSession` es el **punto de entrada** a todo este proceso.

En este entorno, Spark se ejecuta sobre un **clúster Hadoop/YARN real**, compuesto por un nodo maestro (`hadoop-VirtualBox`) y dos nodos trabajadores (`nodo2`, `nodo3`). Al crear la `SparkSession` con `.master("yarn")`, el **driver** se inicia en el nodo maestro —donde corre Jupyter Notebook— y solicita recursos al **ResourceManager** de YARN, que también corre allí. YARN asigna entonces **contenedores** en los **NodeManagers** de `nodo2` y `nodo3`, donde se lanzan los **executors** que ejecutan el trabajo en paralelo. A diferencia del modo local, aquí sí hay un clúster distribuido real: los datos deben residir en HDFS para que todos los nodos puedan acceder a ellos, y el trabajo se reparte físicamente entre máquinas distintas, no entre hilos de una sola CPU.

Esto es importante: **el código PySpark que se escriba aquí es portátil**. La API es exactamente la misma que se usaría en modo local; lo único que cambia es la configuración del `.master(...)` y, en consecuencia, dónde y cómo se ejecuta el trabajo.


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark - Primeros pasos") \
    .master("yarn") \
    .getOrCreate()

# Verificar que la sesión se creó correctamente
print("Versión de Spark:", spark.version)
print("AppName:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)


Versión de Spark: 3.5.7
AppName: PySpark - Primeros pasos
Master: yarn


**Observaciones sobre el código anterior:**
- `.appName(...)`: asigna un nombre descriptivo a la aplicación. En un clúster real, este nombre aparece en la interfaz web de Spark.
- `.master("yarn")`: indica que Spark debe delegar la gestión de recursos a YARN en lugar de ejecutar todo en la misma máquina. El **driver** corre en el nodo maestro (`hadoop-VirtualBox`), y YARN asigna **executors** como contenedores en los nodos trabajadores (`nodo2`, `nodo3`) a través de sus NodeManagers.
- `.getOrCreate()`: obtiene una sesión existente o crea una nueva. Esto evita errores si la celda se ejecuta más de una vez.


## 2. Carga del dataset

Se trabajará con la **Global Power Plant Database** del World Resources Institute (WRI). Este dataset contiene información de aproximadamente 35.000 centrales eléctricas en 167 países, incluyendo su capacidad, tipo de combustible, ubicación geográfica, propietario y generación eléctrica.

**¿Por qué este dataset?**
- Es un dataset real utilizado en análisis energético y de sostenibilidad.
- Combina columnas numéricas (capacidad en MW, generación en GWh) y categóricas (tipo de combustible, país), lo que permite ejercicios variados.
- Su tamaño (aproximadamente 35.000 filas) es pequeño para un clúster, pero suficiente para que las operaciones de Spark tengan sentido pedagógico.

El archivo ya se debe encontrar cargado en el HDFS del clúster (`/datasets/global_power_plant_database.csv`); el primer paso es leerlo directamente desde allí con Spark.


### Lectura del CSV con PySpark

En pandas, la lectura de un CSV se realiza con `pd.read_csv()`. En PySpark, se utiliza `spark.read.csv()` (o su equivalente más general, `spark.read.format("csv").load(...)`).

Dos opciones importantes:
- `header=True`: indica que la primera fila del archivo contiene los nombres de las columnas.
- `inferSchema=True`: le pide a Spark que infiera automáticamente los tipos de datos de cada columna (entero, decimal, texto, etc.). Sin esta opción, **todas las columnas se cargan como texto** (`StringType`).

> **Nota técnica:** En producción, definir el esquema explícitamente (con `StructType`) es preferible a usar `inferSchema`, ya que la inferencia requiere un pase adicional sobre los datos. Para esta guía, `inferSchema=True` es una opción práctica y suficiente.

In [2]:
df = spark.read.csv(
    "hdfs:///datasets/global_power_plant_database.csv",
    header=True,
    inferSchema=True
)

print(f"El DataFrame tiene {df.count()} filas y {len(df.columns)} columnas.")


El DataFrame tiene 34936 filas y 36 columnas.


## 3. Exploración básica del DataFrame

Una vez cargado el dataset, el primer paso es entender su estructura y contenido. A continuación se presentan las operaciones más comunes, con su equivalente en pandas como referencia.

### 3.1. Visualizar las primeras filas: `show()`

Similar a `df.head()` en pandas, `show()` muestra las primeras filas del DataFrame. Por defecto muestra 20 filas; se puede especificar un número diferente. El parámetro `truncate=False` evita que se corten los valores largos.

In [3]:
# Mostrar las primeras 5 filas
df.show(5)

+-------+------------+--------------------+------------+-----------+--------+---------+------------+-----------+-----------+-----------+------------------+-----+----------+--------------------+------------------+-------+---------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+----------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+-----------------------------+------------------------------+------------------------------+------------------------------+------------------------------+------------------------------+
|country|country_long|                name|   gppd_idnr|capacity_mw|latitude|longitude|primary_fuel|other_fuel1|other_fuel2|other_fuel3|commissioning_year|owner|    source|                 url|geolocation_source|wepp_id|year_of_capacity_data|generation_gwh_2013|generation_gwh_2014|generat

In [4]:
# Mostrar sin truncar valores largos (útil para columnas como 'name' u 'owner')
df.select("name", "country_long", "primary_fuel", "capacity_mw").show(5, truncate=False)

+------------------------------------------------+------------+------------+-----------+
|name                                            |country_long|primary_fuel|capacity_mw|
+------------------------------------------------+------------+------------+-----------+
|Kajaki Hydroelectric Power Plant Afghanistan    |Afghanistan |Hydro       |33.0       |
|Kandahar DOG                                    |Afghanistan |Solar       |10.0       |
|Kandahar JOL                                    |Afghanistan |Solar       |10.0       |
|Mahipar Hydroelectric Power Plant Afghanistan   |Afghanistan |Hydro       |66.0       |
|Naghlu Dam Hydroelectric Power Plant Afghanistan|Afghanistan |Hydro       |100.0      |
+------------------------------------------------+------------+------------+-----------+
only showing top 5 rows



### 3.2. Esquema del DataFrame: `printSchema()`

En pandas se usa `df.dtypes` para ver los tipos de datos. En PySpark, `printSchema()` muestra el esquema completo en formato de árbol, incluyendo si cada columna acepta valores nulos (`nullable`).

In [5]:
df.printSchema()

root
 |-- country: string (nullable = true)
 |-- country_long: string (nullable = true)
 |-- name: string (nullable = true)
 |-- gppd_idnr: string (nullable = true)
 |-- capacity_mw: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- primary_fuel: string (nullable = true)
 |-- other_fuel1: string (nullable = true)
 |-- other_fuel2: string (nullable = true)
 |-- other_fuel3: string (nullable = true)
 |-- commissioning_year: double (nullable = true)
 |-- owner: string (nullable = true)
 |-- source: string (nullable = true)
 |-- url: string (nullable = true)
 |-- geolocation_source: string (nullable = true)
 |-- wepp_id: string (nullable = true)
 |-- year_of_capacity_data: integer (nullable = true)
 |-- generation_gwh_2013: double (nullable = true)
 |-- generation_gwh_2014: double (nullable = true)
 |-- generation_gwh_2015: double (nullable = true)
 |-- generation_gwh_2016: double (nullable = true)
 |-- generation_gwh_2017: doubl

### 3.3. Número de filas y columnas

- `df.count()` es equivalente a `len(df)` en pandas. Es una **acción** que recorre todo el DataFrame para contar las filas.
- `df.columns` devuelve la lista de nombres de columnas, similar a `df.columns` en pandas.
- `df.dtypes` devuelve una lista de tuplas `(nombre, tipo)`, similar a `df.dtypes` en pandas.

In [6]:
print("Número de filas:", df.count())
print("Número de columnas:", len(df.columns))
print()
print("Columnas:", df.columns)

Número de filas: 34936
Número de columnas: 36

Columnas: ['country', 'country_long', 'name', 'gppd_idnr', 'capacity_mw', 'latitude', 'longitude', 'primary_fuel', 'other_fuel1', 'other_fuel2', 'other_fuel3', 'commissioning_year', 'owner', 'source', 'url', 'geolocation_source', 'wepp_id', 'year_of_capacity_data', 'generation_gwh_2013', 'generation_gwh_2014', 'generation_gwh_2015', 'generation_gwh_2016', 'generation_gwh_2017', 'generation_gwh_2018', 'generation_gwh_2019', 'generation_data_source', 'estimated_generation_gwh_2013', 'estimated_generation_gwh_2014', 'estimated_generation_gwh_2015', 'estimated_generation_gwh_2016', 'estimated_generation_gwh_2017', 'estimated_generation_note_2013', 'estimated_generation_note_2014', 'estimated_generation_note_2015', 'estimated_generation_note_2016', 'estimated_generation_note_2017']


In [7]:
# Lista de tuplas (nombre_columna, tipo_dato)
df.dtypes

[('country', 'string'),
 ('country_long', 'string'),
 ('name', 'string'),
 ('gppd_idnr', 'string'),
 ('capacity_mw', 'double'),
 ('latitude', 'double'),
 ('longitude', 'double'),
 ('primary_fuel', 'string'),
 ('other_fuel1', 'string'),
 ('other_fuel2', 'string'),
 ('other_fuel3', 'string'),
 ('commissioning_year', 'double'),
 ('owner', 'string'),
 ('source', 'string'),
 ('url', 'string'),
 ('geolocation_source', 'string'),
 ('wepp_id', 'string'),
 ('year_of_capacity_data', 'int'),
 ('generation_gwh_2013', 'double'),
 ('generation_gwh_2014', 'double'),
 ('generation_gwh_2015', 'double'),
 ('generation_gwh_2016', 'double'),
 ('generation_gwh_2017', 'double'),
 ('generation_gwh_2018', 'double'),
 ('generation_gwh_2019', 'double'),
 ('generation_data_source', 'string'),
 ('estimated_generation_gwh_2013', 'double'),
 ('estimated_generation_gwh_2014', 'double'),
 ('estimated_generation_gwh_2015', 'double'),
 ('estimated_generation_gwh_2016', 'double'),
 ('estimated_generation_gwh_2017', 'dou

### 3.4. Estadísticas descriptivas: `describe()`

Análogo a `df.describe()` en pandas, muestra estadísticas resumidas (conteo, media, desviación estándar, mínimo y máximo) de las columnas numéricas. Sin embargo, a diferencia de pandas, en PySpark `describe()` también incluye columnas de texto (mostrando el conteo y valores extremos según orden lexicográfico).

In [8]:
# Estadísticas de las columnas principales
df.select("capacity_mw", "commissioning_year", "primary_fuel", "country_long").describe().show()

[Stage 10:>                                                         (0 + 2) / 2]

+-------+------------------+------------------+------------+------------+
|summary|       capacity_mw|commissioning_year|primary_fuel|country_long|
+-------+------------------+------------------+------------+------------+
|  count|             34936|             17447|       34936|       34936|
|   mean|163.35514790637203|1997.4148229466307|        NULL|        NULL|
| stddev| 489.6360717012953| 23.39783534182863|        NULL|        NULL|
|    min|               1.0|            1896.0|     Biomass| Afghanistan|
|    max|           22500.0|2020.0000000000002|        Wind|    Zimbabwe|
+-------+------------------+------------------+------------+------------+



## 4. Transformaciones fundamentales

En Spark, las **transformaciones** son operaciones que definen *qué se quiere hacer* con los datos, pero **no se ejecutan inmediatamente**. Spark las registra en un plan de ejecución (DAG) y las ejecuta solo cuando se invoca una **acción** (`show()`, `count()`, `collect()`, etc.).

Este comportamiento se denomina **lazy evaluation** (evaluación perezosa). Su principal ventaja es que Spark puede **optimizar** todo el plan antes de ejecutarlo (reordenando operaciones, eliminando pasos innecesarios, etc.).

A continuación se presentan las transformaciones más comunes.

### 4.1. Seleccionar columnas: `select()`

Equivalente a `df[["col1", "col2"]]` en pandas.

In [9]:
# Seleccionar columnas específicas
df_seleccion = df.select("name", "country_long", "primary_fuel", "capacity_mw")

In [10]:
# Hasta aquí, Spark NO ha ejecutado nada. Solo registró la transformación.
# Se necesita una acción para ver los resultados:
df_seleccion.show(5)

+--------------------+------------+------------+-----------+
|                name|country_long|primary_fuel|capacity_mw|
+--------------------+------------+------------+-----------+
|Kajaki Hydroelect...| Afghanistan|       Hydro|       33.0|
|        Kandahar DOG| Afghanistan|       Solar|       10.0|
|        Kandahar JOL| Afghanistan|       Solar|       10.0|
|Mahipar Hydroelec...| Afghanistan|       Hydro|       66.0|
|Naghlu Dam Hydroe...| Afghanistan|       Hydro|      100.0|
+--------------------+------------+------------+-----------+
only showing top 5 rows



### 4.2. Filtrar filas: `filter()` / `where()`

Equivalente a `df[df["columna"] == valor]` en pandas. En PySpark, `filter()` y `where()` son sinónimos.

In [11]:
from pyspark.sql.functions import col

# Filtrar plantas en Colombia
df_colombia = df.filter(col("country") == "COL")
df_colombia.select("name", "primary_fuel", "capacity_mw", "owner").show(10, truncate=False)

+---------------------+------------+-----------+-------------+
|name                 |primary_fuel|capacity_mw|owner        |
+---------------------+------------+-----------+-------------+
|Calderas             |Hydro       |26.0       |Isagen S.A.  |
|Cantral La Tasajera  |Hydro       |306.0      |EPM          |
|Cartagena            |Oil         |188.0      |ENDESA       |
|Central Guadalupe III|Hydro       |270.0      |EPM          |
|Central Guatapé      |Hydro       |560.0      |EPM          |
|Chivor               |Hydro       |1000.0     |AES Chivor   |
|El Paso              |Solar       |86.2       |NULL         |
|Gavio                |Hydro       |1200.0     |Emgesa       |
|Gecelca power station|Coal        |164.0      |Gecelca SAESP|
|Jaguas               |Hydro       |170.0      |Isagen S.A.  |
+---------------------+------------+-----------+-------------+
only showing top 10 rows



In [12]:
# Filtrar plantas solares con capacidad mayor a 100 MW
df_solar_grande = df.filter(
    (col("primary_fuel") == "Solar") & (col("capacity_mw") > 100)
)

print(f"Plantas solares con más de 100 MW: {df_solar_grande.count()}")
df_solar_grande.select("name", "country_long", "capacity_mw").show(10, truncate=False)

Plantas solares con más de 100 MW: 227
+-------------------------+------------+-----------+
|name                     |country_long|capacity_mw|
+-------------------------+------------+-----------+
|Hauts Plateaux Centre    |Algeria     |106.6      |
|Cauchari                 |Argentina   |300.0      |
|Bungala I                |Australia   |110.0      |
|Bungala II               |Australia   |110.0      |
|Clare Solar Farm         |Australia   |130.0      |
|Darling Downs Solar Farm |Australia   |110.0      |
|Haughton River Solar Farm|Australia   |500.0      |
|Hayman Solar Farm        |Australia   |180.0      |
|Karadoc Solar Farm       |Australia   |112.0      |
|Nyngan Solar Plant       |Australia   |102.0      |
+-------------------------+------------+-----------+
only showing top 10 rows



### 4.3. Agregar o modificar columnas: `withColumn()`

Equivalente a `df["nueva_col"] = expresión` en pandas. En PySpark, el DataFrame es **inmutable**: `withColumn()` no modifica el DataFrame original, sino que retorna uno nuevo.

In [13]:
from pyspark.sql.functions import col, round as spark_round

# Crear columna con la capacidad en GW (1 GW = 1000 MW)
df_con_gw = df.withColumn("capacity_gw", spark_round(col("capacity_mw") / 1000, 4))

df_con_gw.select("name", "capacity_mw", "capacity_gw").show(5)

+--------------------+-----------+-----------+
|                name|capacity_mw|capacity_gw|
+--------------------+-----------+-----------+
|Kajaki Hydroelect...|       33.0|      0.033|
|        Kandahar DOG|       10.0|       0.01|
|        Kandahar JOL|       10.0|       0.01|
|Mahipar Hydroelec...|       66.0|      0.066|
|Naghlu Dam Hydroe...|      100.0|        0.1|
+--------------------+-----------+-----------+
only showing top 5 rows



### 4.4. Renombrar columnas: `withColumnRenamed()`

Equivalente a `df.rename(columns={"viejo": "nuevo"})` en pandas.

In [14]:
# Renombrar 'primary_fuel' a 'combustible_principal'
df_renombrado = df.withColumnRenamed("primary_fuel", "combustible_principal")

df_renombrado.select("name", "combustible_principal", "capacity_mw").show(5)

+--------------------+---------------------+-----------+
|                name|combustible_principal|capacity_mw|
+--------------------+---------------------+-----------+
|Kajaki Hydroelect...|                Hydro|       33.0|
|        Kandahar DOG|                Solar|       10.0|
|        Kandahar JOL|                Solar|       10.0|
|Mahipar Hydroelec...|                Hydro|       66.0|
|Naghlu Dam Hydroe...|                Hydro|      100.0|
+--------------------+---------------------+-----------+
only showing top 5 rows



### 4.5. Eliminar columnas: `drop()`

Equivalente a `df.drop(columns=[...])` en pandas.

In [15]:
# Eliminar columnas de metadatos que no se necesitan para el análisis
df_limpio = df.drop("url", "geolocation_source", "wepp_id", "source")

print("Columnas originales:", len(df.columns))
print("Columnas después de drop:", len(df_limpio.columns))

Columnas originales: 36
Columnas después de drop: 32


### 4.6. Valores únicos: `distinct()`

Equivalente a `df["col"].unique()` en pandas.

In [16]:
# ¿Cuántos tipos de combustible existen en el dataset?
tipos_combustible = df.select("primary_fuel").distinct()

print(f"Tipos de combustible distintos: {tipos_combustible.count()}")
tipos_combustible.show(20, truncate=False)

Tipos de combustible distintos: 15
+--------------+
|primary_fuel  |
+--------------+
|Wind          |
|Petcoke       |
|Waste         |
|Solar         |
|Oil           |
|Other         |
|Nuclear       |
|Geothermal    |
|Hydro         |
|Cogeneration  |
|Storage       |
|Wave and Tidal|
|Gas           |
|Biomass       |
|Coal          |
+--------------+



### 4.7. Lazy evaluation en la práctica

Para evidenciar el comportamiento perezoso, se construirá una cadena de transformaciones y se observará que **nada se ejecuta** hasta que se invoque una acción.

El siguiente código encadena tres transformaciones. Al ejecutar la celda, Spark simplemente registra el plan y termina de inmediato.

In [17]:
# Encadenar transformaciones: Spark NO ejecuta nada aquí
df_resultado = (
    df
    .filter(col("primary_fuel") == "Wind")
    .select("name", "country_long", "capacity_mw", "commissioning_year")
    .withColumn("decada", (col("commissioning_year") / 10).cast("int") * 10)
)

# Si se imprime df_resultado, solo se ve la definición del DataFrame, no datos
print(type(df_resultado))
print("La celda terminó, pero Spark NO leyó ni procesó datos todavía.")

<class 'pyspark.sql.dataframe.DataFrame'>
La celda terminó, pero Spark NO leyó ni procesó datos todavía.


In [18]:
# Ahora sí, al invocar la ACCIÓN show(), Spark ejecuta toda la cadena
df_resultado.show(10)

+--------------------+------------+-----------+------------------+------+
|                name|country_long|capacity_mw|commissioning_year|decada|
+--------------------+------------+-----------+------------------+------+
|         Ross Island|  Antarctica|        1.0|              NULL|  NULL|
|COMODORO RIVADAVI...|   Argentina|      16.56|              NULL|  NULL|
|        GENERAL ACHA|   Argentina|        1.8|              NULL|  NULL|
|    MAYOR BURATOVICH|   Argentina|       2.02|              NULL|  NULL|
|PARQUE EOLICO ARA...|   Argentina|       25.2|              NULL|  NULL|
|PARQUE EOLICO ARA...|   Argentina|       25.2|              NULL|  NULL|
|PARQUE EOLICO DIA...|   Argentina|        6.3|              NULL|  NULL|
|PARQUE EOLICO EL ...|   Argentina|        3.0|              NULL|  NULL|
|PARQUE EOLICO LOM...|   Argentina|       50.0|              NULL|  NULL|
|PARQUE EOLICO RAW...|   Argentina|       48.6|              NULL|  NULL|
+--------------------+------------+---

**Conclusión:** Las transformaciones (`filter`, `select`, `withColumn`) solo definen el plan. La acción (`show`) dispara la ejecución real. Este patrón se repetirá constantemente al trabajar con Spark.

## 5. Acciones fundamentales

Las **acciones** son operaciones que disparan la ejecución del plan de transformaciones y devuelven un resultado al driver. A diferencia de las transformaciones, las acciones **no retornan un DataFrame**, sino valores concretos (filas, números, listas).

In [19]:
# count(): retorna el número total de filas (un entero)
total = df.count()
print(f"Total de plantas: {total}")

Total de plantas: 34936


In [20]:
# first(): retorna la primera fila como un objeto Row
primera_fila = df.first()
print(primera_fila)
print()
# Se puede acceder a los campos por nombre
print(f"Nombre: {primera_fila['name']}, País: {primera_fila['country_long']}")

Row(country='AFG', country_long='Afghanistan', name='Kajaki Hydroelectric Power Plant Afghanistan', gppd_idnr='GEODB0040538', capacity_mw=33.0, latitude=32.322, longitude=65.119, primary_fuel='Hydro', other_fuel1=None, other_fuel2=None, other_fuel3=None, commissioning_year=None, owner=None, source='GEODB', url='http://globalenergyobservatory.org', geolocation_source='GEODB', wepp_id='1009793', year_of_capacity_data=2017, generation_gwh_2013=None, generation_gwh_2014=None, generation_gwh_2015=None, generation_gwh_2016=None, generation_gwh_2017=None, generation_gwh_2018=None, generation_gwh_2019=None, generation_data_source=None, estimated_generation_gwh_2013=123.77, estimated_generation_gwh_2014=162.9, estimated_generation_gwh_2015=97.39, estimated_generation_gwh_2016=137.76, estimated_generation_gwh_2017=119.5, estimated_generation_note_2013='HYDRO-V1', estimated_generation_note_2014='HYDRO-V1', estimated_generation_note_2015='HYDRO-V1', estimated_generation_note_2016='HYDRO-V1', estim

In [21]:
# take(n): retorna las primeras n filas como una lista de objetos Row
tres_filas = df.take(3)
for fila in tres_filas:
    print(f"{fila['name']} — {fila['primary_fuel']} — {fila['capacity_mw']} MW")

Kajaki Hydroelectric Power Plant Afghanistan — Hydro — 33.0 MW
Kandahar DOG — Solar — 10.0 MW
Kandahar JOL — Solar — 10.0 MW


In [29]:
# collect(): retorna TODAS las filas como una lista de Row
# ⚠️ PRECAUCIÓN: collect() trae TODOS los datos al driver.
# En datasets grandes (millones de filas), esto puede causar errores de memoria.
# Aquí se usa sobre un DataFrame filtrado y pequeño para ilustrar su uso.

plantas_nucleares_col = df.filter(
    (col("primary_fuel") == "Hydro") & (col("country") == "USA")
)

resultado = plantas_nucleares_col.collect()
print(f"Plantas nucleares en Colombia: {len(resultado)}")
# Colombia no tiene plantas nucleares, así que la lista estará vacía

Plantas nucleares en Colombia: 1449


**Resumen de acciones:**

| Acción | Retorna | Equivalente en pandas |
|--------|---------|----------------------|
| `show(n)` | Imprime n filas (no retorna valor) | `df.head(n)` |
| `count()` | Entero con el total de filas | `len(df)` |
| `first()` | Un objeto `Row` | `df.iloc[0]` |
| `take(n)` | Lista de n objetos `Row` | `df.head(n).values.tolist()` |
| `collect()` | Lista con **todas** las filas | `df.values.tolist()` |

---

En la Sesión 21 se retomará este mismo dataset: la SparkSession y la lectura del DataFrame se recrean al inicio de ese notebook, ya que se ejecutará en un kernel nuevo.
